## Overview

1. Clustering               pair shortlisting, Dinov2
2. Local matching           SuperPoint + SuperGlue, or MASt3R (mby later lightglue instead of superglue)
3. Geometric verification   RANSAC 
4. Reconstruction           COLMAP/pycolmap structure-from-motion (SfM) → camera poses
                            each separate reconstruction = one scene cluster; unregistered = outlier


### First simple pipeline

DINOv2 embedding per image -> clustering by HDBSCAN(cosine) -> Create pairs within each cluster -> Keypoint extraction per image via SuperPoint -> Match Keypointlist per pair with SuperGlue -> 

## Pair shortlisting

Pair sortlisting via embedding using Dinov2
https://github.com/facebookresearch/dinov2

embed Images and group them into sets of similar images,
return a list of candidate pairs to match

In [1]:
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from pathlib import Path
import sys
import cv2
import numpy as np


from lightglue import SuperPoint, LightGlue
from lightglue.utils import load_image, rbd
import pandas as pd
from tqdm import tqdm

ROOT = Path('image-matching-challenge-2025/train')

## cd IMC2025-ML-Project
## git clone https://github.com/magicleap/SuperGluePretrainedNetwork.git


dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

Using cache found in /Users/lucbaumeler/.cache/torch/hub/facebookresearch_dinov2_main
/Users/lucbaumeler/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/lucbaumeler/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/lucbaumeler/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
##Hyperparams
k_neighbors = 20
coarse = 512
fine = 2048
coarse_size = 512
fine_size = 1024


In [3]:
dinov2.eval().to(device)

transform = T.Compose([T.Resize((448, 448)), T.ToTensor(), T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])



#### Example embeddings with cosine similarity

In [4]:
x1 = Image.open('image-matching-challenge-2025/test/stairs/stairs_split_1_1710453576271.png').convert('RGB')
x2 = Image.open('image-matching-challenge-2025/test/stairs/stairs_split_1_1710453601885.png').convert('RGB')
x3 = Image.open('image-matching-challenge-2025/test/ETs/another_et_another_et001.png').convert('RGB')

x1 = transform(x1).unsqueeze(0).to(device)
x2 = transform(x2).unsqueeze(0).to(device)
x3 = transform(x3).unsqueeze(0).to(device)



with torch.no_grad():
    y1 = dinov2(x1)
    y2 = dinov2(x2)
    y3 = dinov2(x3)
    
sim = F.cosine_similarity(y1, y2)
sim2 = F.cosine_similarity(y1, y3)
print(sim.item())
print(sim2.item())


0.8893314599990845
0.02078629657626152


**Create list of image embeddings and per folder**

In [5]:
ds = ImageFolder('image-matching-challenge-2025/train', transform)
loader = DataLoader(dataset=ds, batch_size=64, shuffle=False)

embs = []
labls = []

with torch.no_grad():
    for imgs, labels in loader:
        embs.append(dinov2(imgs.to(device)))
        labls.append(labels.to(device))

In [6]:
print(embs[0])
print(labls[0])

tensor([[ 0.6501, -3.5805, -1.6924,  ...,  1.5323, -0.8058, -1.3643],
        [-0.9998, -3.7093, -0.3785,  ...,  0.5889,  0.0951, -1.5165],
        [-0.8266, -3.5939, -1.2035,  ...,  1.7604,  1.4490, -1.6209],
        ...,
        [ 2.5760, -0.2871, -1.6255,  ...,  0.5779,  2.3726,  3.6004],
        [ 1.4662, -0.4711, -2.8981,  ...,  1.2748,  3.1438,  3.2336],
        [ 1.3754, -0.6801, -1.5857,  ...,  2.2001,  3.8991,  2.8979]],
       device='mps:0')
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='mps:0')


In [7]:
E = torch.cat(embs)
L = torch.cat(labls)
emb = {name: E[L == idx] for name, idx in ds.class_to_idx.items()}
paths = {name: [p for p, l in ds.samples if l == idx] for name, idx in ds.class_to_idx.items()}

for name, X in emb.items():
    print(name, X.shape, len(paths[name]))

ETs torch.Size([22, 384]) 22
amy_gardens torch.Size([200, 384]) 200
fbk_vineyard torch.Size([163, 384]) 163
imc2023_haiper torch.Size([54, 384]) 54
imc2023_heritage torch.Size([209, 384]) 209
imc2023_theather_imc2024_church torch.Size([76, 384]) 76
imc2024_dioscuri_baalshamin torch.Size([138, 384]) 138
imc2024_lizard_pond torch.Size([214, 384]) 214
pt_brandenburg_british_buckingham torch.Size([225, 384]) 225
pt_piazzasanmarco_grandplace torch.Size([168, 384]) 168
pt_sacrecoeur_trevi_tajmahal torch.Size([225, 384]) 225
pt_stpeters_stpauls torch.Size([200, 384]) 200
stairs torch.Size([51, 384]) 51


In [8]:
from sklearn.neighbors import NearestNeighbors

def knn_pairs(X, files, k=k_neighbors):
    n = len(files)
    k = min(k, n-1)
    X = F.normalize(X, dim=1).cpu().numpy()

    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine').fit(X)
    _, neigh = nn.kneighbors(X)
    idx_pairs = sorted({(min(i, j), max(i, j)) for i in range(n) for j in neigh[i] if j != i})
    file_pairs = [tuple(sorted((files[i], files[j]))) for i, j in idx_pairs]

    return file_pairs, idx_pairs

pairs, pair_idx = {}, {}
for name, X in emb.items():
    files = [Path(p).name for p in paths[name]]
    pairs[name], pair_idx[name] = knn_pairs(X, files, k=k_neighbors)


### Keypoint extraction + Matching



In [ ]:

coarse_ex = SuperPoint(max_num_keypoints=coarse).eval().to(device)
fine_ex = SuperPoint(max_num_keypoints=fine).eval().to(device)
matcher = LightGlue(features='superpoint').eval().to(device)

def extract_all(extractor, name, files, resize):
    feats = {}
    for f in tqdm(files, desc=f'extract {name}', leave=False):
        img = load_image(ROOT / name / f).to(device)          
        feats[f] = extractor.extract(img, resize=resize)
    return feats
    
keypoints = []                                     

with torch.no_grad():
    for name in pairs:
        files = sorted({f for pair in pairs[name] for f in pair})
        feats = extract_all(coarse_ex, name, files, coarse_size)

        for a, b in tqdm(pairs[name], desc=f'match {name}'):
            out = rbd(matcher({'image0': feats[a], 'image1': feats[b]}))
            idx = out['matches'].cpu().numpy()                    
            kA = feats[a]['keypoints'][0].cpu().numpy()            
            kB = feats[b]['keypoints'][0].cpu().numpy()            

            matches0 = np.full(len(kA), -1, dtype=np.int64)       
            matches0[idx[:, 0]] = idx[:, 1]

            keypoints.append({'dataset': name, 'scene': None, 'pair': (a, b),
                              'keypoints0': kA, 'keypoints1': kB,
                              'matches0': matches0,
                              'scale': max(feats[a]['image_size'].max().item(),
                                           feats[b]['image_size'].max().item()) / coarse_size})

match imc2024_dioscuri_baalshamin:  93%|█████████▎| 1765/1903 [06:12<00:36,  3.78it/s]

In [ ]:
# Geometric verification using RANSAC

ransac_results = []

# RANSAC parameters
ransac_threshold = 1.0  # Maximum epipolar error in pixels
ransac_confidence = 0.999

for result in keypoints:
    # Keypoints detected in the two images
    kp0 = result['keypoints0']
    kp1 = result['keypoints1']

    # matches0[i] gives the index of the matching keypoint in image 1.
    # A value of -1 means that keypoint i has no match.
    matches0 = result['matches0']

    valid = matches0 >= 0
    matched_indices0 = np.flatnonzero(valid)
    matched_indices1 = matches0[valid]

    pts0 = kp0[matched_indices0].astype(np.float32)
    pts1 = kp1[matched_indices1].astype(np.float32)

    # A fundamental matrix needs enough corresponding points.
    if len(pts0) < 8:
        ransac_results.append({
            **result,
            'fundamental_matrix': None,
            'inlier_mask': np.zeros(len(pts0), dtype=bool),
            'inlier_points0': np.empty((0, 2), dtype=np.float32),
            'inlier_points1': np.empty((0, 2), dtype=np.float32),
            'num_matches': len(pts0),
            'num_inliers': 0,
            'inlier_ratio': 0.0,
            'verified': False
        })
        continue

    # Estimate the fundamental matrix and identify geometric inliers.
    F_matrix, mask = cv2.findFundamentalMat(
        pts0,
        pts1,
        method=cv2.FM_RANSAC,
        ransacReprojThreshold=ransac_threshold * result['scale'],
        confidence=ransac_confidence
    )

    # Handle cases where estimation fails.
    if F_matrix is None or mask is None or F_matrix.shape != (3, 3):
        inliers = np.zeros(len(pts0), dtype=bool)
        F_matrix = None
    else:
        inliers = mask.ravel().astype(bool)

    num_matches = len(pts0)
    num_inliers = int(inliers.sum())
    inlier_ratio = num_inliers / num_matches if num_matches else 0.0

    # Save the geometrically consistent correspondences.
    ransac_results.append({
        **result,
        'fundamental_matrix': F_matrix,
        'inlier_mask': inliers,
        'inlier_points0': pts0[inliers],
        'inlier_points1': pts1[inliers],
        'num_matches': num_matches,
        'num_inliers': num_inliers,
        'inlier_ratio': inlier_ratio,
        'verified': num_inliers >= 8
    })

print(f"Processed {len(ransac_results)} image pairs.")

# Display a few results
for result in ransac_results[:10]:
    print(
        f"{result['dataset']} / {result['scene']} | "
        f"{result['pair'][0]} <-> {result['pair'][1]} | "
        f"matches: {result['num_matches']} | "
        f"inliers: {result['num_inliers']} | "
        f"ratio: {result['inlier_ratio']:.2%}"
    )


Processed 1 image pairs.
stairs / cluster2 | a <-> b | matches: 6 | inliers: 0 | ratio: 0.00%


In [ ]:
# Filter image pairs using RANSAC statistics

# Hyperparameters
min_matches = 15
min_inliers = 12
min_inlier_ratio = 0.20

# Keep pairs satisfying all three conditions
filtered_pairs = [
    result for result in ransac_results
    if result["num_matches"] >= min_matches
    and result["num_inliers"] >= min_inliers
    and result["inlier_ratio"] >= min_inlier_ratio
]

# Summary
print(f"Total pairs before filtering: {len(ransac_results)}")
print(f"Pairs retained: {len(filtered_pairs)}")
print(f"Pairs rejected: {len(ransac_results) - len(filtered_pairs)}")

if ransac_results:
    print(
        f"Retention rate: "
        f"{len(filtered_pairs) / len(ransac_results):.1%}"
    )

# Display the strongest retained pairs
filtered_pairs_sorted = sorted(
    filtered_pairs,
    key=lambda r: (r["num_inliers"], r["inlier_ratio"]),
    reverse=True
)

print("\nTop retained pairs:")

for result in filtered_pairs_sorted[:15]:
    print(
        f'{result["dataset"]} | '
        f'{result["pair"][0]} <-> {result["pair"][1]} | '
        f'Matches: {result["num_matches"]} | '
        f'Inliers: {result["num_inliers"]} | '
        f'Ratio: {result["inlier_ratio"]:.1%}'
    )


In [ ]:
import networkx as nx

image_graph = nx.Graph()

for result in filtered_pairs:
    dataset = result["dataset"]
    scene = result["scene"]
    image_a, image_b = result["pair"]

    node_a = (dataset, image_a)
    node_b = (dataset, image_b)

    kp0 = np.asarray(result["keypoints0"], dtype=np.float32)
    kp1 = np.asarray(result["keypoints1"], dtype=np.float32)
    matches0 = np.asarray(result["matches0"], dtype=np.int32)
    inlier_mask = np.asarray(result["inlier_mask"], dtype=bool)

    # Recover the original SuperPoint indices for valid matches.
    matched_indices0 = np.flatnonzero(matches0 >= 0)
    matched_indices1 = matches0[matched_indices0]

    if len(inlier_mask) != len(matched_indices0):
        raise ValueError(
            f"Inlier mask mismatch for pair {(image_a, image_b)}"
        )

    # Keep only RANSAC inliers, expressed as original keypoint indices.
    inlier_matches = np.column_stack((
        matched_indices0[inlier_mask],
        matched_indices1[inlier_mask]
    )).astype(np.uint32)

    # Store each image's original keypoints once.
    for node, keypoints_array in (
        (node_a, kp0),
        (node_b, kp1),
    ):
        if node not in image_graph:
            image_graph.add_node(
                node,
                dataset=dataset,
                scene=scene,
                image_name=node[1],
                keypoints=keypoints_array,
            )
        else:
            existing = image_graph.nodes[node]["keypoints"]

            # The same image must always use the same keypoint array
            # and therefore the same feature indices.
            if (
                existing.shape != keypoints_array.shape
                or not np.allclose(existing, keypoints_array, atol=1e-4)
            ):
                raise ValueError(
                    f"Inconsistent keypoints for image {node}. "
                    "Check feature caching and image ordering."
                )

    # Preserve the original pair order explicitly.
    # inlier_matches[:, 0] indexes keypoints0 for image_a;
    # inlier_matches[:, 1] indexes keypoints1 for image_b.
    image_graph.add_edge(
        node_a,
        node_b,
        image0=node_a,
        image1=node_b,
        num_matches=int(result["num_matches"]),
        num_inliers=int(result["num_inliers"]),
        inlier_ratio=float(result["inlier_ratio"]),
        fundamental_matrix=result["fundamental_matrix"],
        inlier_matches=inlier_matches,
    )

for name in pairs:
    for f in {f for pair in pairs[name] for f in pair}:
        if (name, f) not in image_graph:
            image_graph.add_node((name, f), dataset=name, scene=None, image_name=f, keypoints=None)

print("COLMAP-ready image graph built.")
print(f"Images (nodes): {image_graph.number_of_nodes()}")
print(f"Verified pairs (edges): {image_graph.number_of_edges()}")
print(f"Connected components: {nx.number_connected_components(image_graph)}")


In [ ]:
# sort out the connected components

components = sorted(
    nx.connected_components(image_graph),
    key=len,
    reverse=True
)

print("Connected components:", len(components))

for i, component in enumerate(components[:10], start=1):
    print(f"\nComponent {i}: {len(component)} images")
    for node in sorted(component):
        print(" ", node)
